# Qwen3.5-9B: Few-shot grading

- Text-only grading with an 80/10/10 split by unique question, using seed 42.
- Each prompt contains the question, student answer, and excellent, average, and poor example answers with their recorded scores.
- The requested response is a JSON object containing a numeric `score` and string `feedback`.
- Criterion arrays are retained as dataset metadata and as the supervised training completion, but inference validates the scalar score object.
- Optional QLoRA training uses four validation/save checks per epoch, plateau learning-rate reduction, early stopping, and best-checkpoint restoration.
- Thinking is disabled for tokenizer templates and generation.
- Data preparation and validation run by default; set `RUN_TRAINING=True` to load the model, train, save, and run local inference.
- The held-out set is evaluated with the trained adapter and a separate few-shot baseline when the corresponding execution flags are enabled.

## Install dependencies

- Installs the CUDA compiler/toolchain and the Python packages used by this notebook.
- The CUDA compiler setup targets the Linux/Conda training environment; optional CUDA kernels can be installed separately.
- The cell configures compiler and CUDA environment variables before model imports.

In [1]:
# %pip install -y -c conda-forge -c nvidia gcc_linux-64 cuda-nvcc=13.0
# %pip install -U torch transformers accelerate peft datasets bitsandbytes trl sentencepiece pandas scikit-learn ninja packaging huggingface_hub python-dotenv llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import os
import shutil
import subprocess
import sys

# Dependencies are installed in the project virtual environment before opening this notebook.
cuda_home = os.environ.get("CUDA_HOME")
if cuda_home:
    cuda_include = os.path.join(cuda_home, "targets", "x86_64-linux", "include")
    os.environ["PATH"] = os.path.join(cuda_home, "bin") + os.pathsep + os.environ.get("PATH", "")
    os.environ["CPATH"] = cuda_include + os.pathsep + os.environ.get("CPATH", "")
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "8.9")
os.environ.setdefault("MAX_JOBS", "1")
if (conda_cc := shutil.which("x86_64-conda-linux-gnu-cc")):
    os.environ.setdefault("CC", conda_cc)
if (conda_cxx := shutil.which("x86_64-conda-linux-gnu-c++")):
    os.environ.setdefault("CXX", conda_cxx)

print("Environment setup complete. Install optional CUDA kernels separately if desired.")

Environment setup complete. Install optional CUDA kernels separately if desired.


## Imports and hardware

- Imports the dataset, tokenizer, PEFT, Transformers, TRL, metrics, and utility libraries used below.
- Reports PyTorch, CUDA, compiler, and optional `causal_conv1d`/`fla` availability.
- Dataset checks can run on CPU; QLoRA model loading and training require a CUDA GPU.

In [2]:
import ast
import csv
import gc
import importlib.util
import json
import math
import os
import random
import re
import shutil
import requests

# Set the compiler before importing PyTorch/Triton.
cc = os.environ.get("CC") or next((
    path for name in (
        "x86_64-conda-linux-gnu-cc", "x86_64-conda_cos7-linux-gnu-cc", "gcc", "clang"
    ) if (path := shutil.which(name))
), None)
if cc:
    os.environ["CC"] = cc

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_recall_fscore_support
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
from trl import SFTConfig, SFTTrainer
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
for package in ("causal_conv1d", "fla"):
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")

PyTorch: 2.14.1+cu130 | CUDA: True | Compiler: /opt/apps/software/GCCcore/13.3.0/bin/gcc
causal_conv1d installed: True
fla installed: True
GPU: NVIDIA A40 | VRAM: 44.4 GiB


## Configuration

- Defines the model, dataset, output directory, seeds, LoRA settings, optimizer settings, and generation limits.
- `MAX_SEQ_LENGTH=None` sizes the sequence limit from the longest formatted training/validation example, rounded up to 32 tokens.
- An explicit sequence-length cap rejects overlength training/validation rows instead of silently truncating them.
- `TRAIN_BATCH_SIZE=1` with gradient accumulation of 8 gives an effective batch size of 8 on one GPU.
- `RUN_TRAINING` gates model loading, adapter training, saving, and local inference.
- `RUN_TEST_EVALUATION` and `RUN_HF_UPLOAD` control later evaluation and upload steps; uploads remain disabled by default.

In [3]:
MODEL_ID = "Qwen/Qwen3.5-9B"
DATASET_PATH = "../Datasets/training_data_moh_eng_fewshot_v2.csv"
OUTPUT_DIR = "../qwen35-9b-grading-lora-moh-eng-fewshot"
RUN_TRAINING = False
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = False  # Enable only after local evaluation succeeds.
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
evaluation_completed = False
SEED = 2026
SPLIT_SEED = 2026
MAX_SEQ_LENGTH = None  # Auto-size from formatted training/validation data.

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size 8 on one GPU.
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKENS = 128
GENERATION_TOKENS_PER_CRITERION = 8
FEWSHOT_MAX_NEW_TOKENS = 200

set_seed(SEED)

## Load the dataset

- Reads the CSV with `keep_default_na=False`, so blank values and literal `NA` text are preserved.
- Requires the question, answer, example-answer, criterion-array, and recorded-score columns used by later cells.
- Rejects missing required columns and empty datasets, then displays the first rows.

In [4]:
required_columns = [
    "id",
    "question",
    "desired_answer",
    "student_answer",
    "max_score",
    "student_score",
    "examples"
]
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())

Rows: 3778


,id,question,desired_answer,student_answer,max_score,student_score,examples
0,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,High risk problems are address in the prototyp...,1,0,"[""Score 1/1: A prototype is a stripped-down pr..."
1,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,To simulate portions of the desired final prod...,1,1,"[""Score 1/1: A prototype is a stripped-down pr..."
2,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,A prototype program simulates the behaviors of...,1,1,"[""Score 1/1: A prototype is a stripped-down pr..."
3,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,Defined in the Specification phase a prototype...,1,1,"[""Score 1/1: A prototype is a stripped-down pr..."
4,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,It is used to let the users have a first idea ...,1,0,"[""Score 1/1: A prototype is a stripped-down pr..."


## Parse and validate rows

- Parses JSON/Python arrays and legacy bracketed text, including criteria containing commas.
- Requires nonempty questions, desired answers, example answers, and criteria; aligned criterion arrays; binary criterion labels; and finite nonnegative score metadata.
- Preserves blank student answers and recorded final/example scores.
- Reports rows whose recorded final score differs from the weighted criterion total without replacing the recorded score.

In [5]:
EXAMPLE_RE = re.compile(r"^Score\s+(\d+)\s*/\s*(\d+):\s*(.+)$", re.DOTALL)


def parse_int(value, column_name, minimum=0, maximum=None):
    try:
        result = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{column_name} must be numeric: {value!r}") from error

    if not math.isfinite(result) or not result.is_integer():
        raise ValueError(f"{column_name} must be a whole number: {value!r}")

    result = int(result)
    if result < minimum or (maximum is not None and result > maximum):
        raise ValueError(
            f"{column_name} must be between {minimum} and {maximum}: {value!r}"
        )
    return result


def parse_examples(value, max_score):
    """Parse the examples cell into [(score, answer), ...] and check it matches max_score."""
    if isinstance(value, list):
        items = value
    else:
        if not isinstance(value, str):
            raise TypeError(f"examples must be a JSON array string, got {type(value).__name__}.")
        try:
            items = json.loads(value)
        except json.JSONDecodeError as error:
            raise ValueError(f"examples is not valid JSON: {value[:100]!r}") from error

    if not isinstance(items, list):
        raise ValueError(f"examples must be a list, got {type(items).__name__}.")

    if len(items) != max_score + 1:
        raise ValueError(
            f"Expected {max_score + 1} examples for max_score={max_score}, got {len(items)}."
        )

    parsed = []
    for item in items:
        m = EXAMPLE_RE.match(str(item).strip())
        if not m:
            raise ValueError(f"Example not in 'Score X/Y: answer' format: {item!r}")
        score, out_of, answer = int(m.group(1)), int(m.group(2)), m.group(3).strip()
        if out_of != max_score:
            raise ValueError(f"Example labelled /{out_of} but max_score is {max_score}.")
        if not answer:
            raise ValueError("Example answer is empty.")
        parsed.append((score, answer))

    scores = sorted(score for score, _ in parsed)
    if scores != list(range(max_score + 1)):
        raise ValueError(f"Example scores should cover 0..{max_score} once each, got {scores}.")

    # Highest score first
    return sorted(parsed, key=lambda x: -x[0])


records = []
errors = []

for index, row in enumerate(df.to_dict("records")):
    try:
        question = str(row["question"]).strip()
        desired_answer = str(row["desired_answer"]).strip()
        student_answer = row["student_answer"]
        student_answer = "" if student_answer is None or (
            isinstance(student_answer, float) and math.isnan(student_answer)
        ) else str(student_answer)

        if not question:
            raise ValueError("Question must be nonempty.")
        if not desired_answer:
            raise ValueError("Desired answer must be nonempty.")

        max_score = parse_int(row["max_score"], "max_score", minimum=1)
        student_score = parse_int(
            row["student_score"], "student_score", minimum=0, maximum=max_score
        )
        examples = parse_examples(row["examples"], max_score)

        records.append({
            "row_index": index,
            "id": str(row["id"]),
            "question": question,
            "desired_answer": desired_answer,
            "student_answer": student_answer,
            "max_score": max_score,
            "student_score": student_score,
            "examples": examples,  # [(score, answer), ...] highest first
        })

    except (KeyError, ValueError, TypeError) as error:
        errors.append(f"Row {index}: {type(error).__name__}: {error}")

if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))

print(f"All {len(records)} rows passed validation.")

empty_answers = sum(not r["student_answer"].strip() for r in records)
print("Rows with an empty student answer:", empty_answers)
print("Score distribution:", pd.Series([r["student_score"] for r in records]).value_counts().sort_index().to_dict())

All 3778 rows passed validation.
Rows with an empty student answer: 1
Score distribution: {0: 908, 1: 1492, 2: 936, 3: 382, 4: 57, 5: 3}


## Grading prompt

- Defines a strict system prompt that asks for only a JSON object with a numeric `score` and string `feedback`.
- The prompt compares the student answer with excellent, average, and poor examples and their scores.

In [6]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Compare the student answer holistically against the example answers provided,\n"
    "which show what score they should receive.\n"
    "Return ONLY a valid JSON object with a numeric score. e.g. {\"score\": 1}\n"
    "Do not include explanations or markdown outside the JSON object."
)

print(SYSTEM_PROMPT)

You are a strict grading assistant.

Compare the student answer holistically against the example answers provided,
which show what score they should receive.
Return ONLY a valid JSON object with a numeric score. e.g. {"score": 1}
Do not include explanations or markdown outside the JSON object.


## Build examples

- Builds a structured prompt from the three scored example bands, question, and student answer.
- Retains all parsed fields for evaluation and creates a separate assistant completion containing the criterion-score array used for supervised training.

In [7]:
EXAMPLE_RE = re.compile(r"^Score\s+(\d+)\s*/\s*(\d+):\s*(.*)$", re.DOTALL)

def parse_examples(raw):
    """Turn the examples cell into [(score, answer), ...], highest score first."""
    parsed = []
    for item in json.loads(raw):
        m = EXAMPLE_RE.match(item.strip())
        if not m:
            raise ValueError(f"Unexpected example format: {item!r}")
        parsed.append((int(m.group(1)), m.group(3).strip()))
    return sorted(parsed, key=lambda x: -x[0])

def _normalize_examples(examples):
    out = []
    for ex in examples:
        if isinstance(ex, str):
            m = EXAMPLE_RE.match(ex.strip())
            out.append((int(m.group(1)), m.group(3).strip()))
        else:
            out.append((int(ex[0]), str(ex[1])))
    return sorted(out, key=lambda x: -x[0])

def make_prompt(row):
    max_score = int(row["max_score"])
    examples_block = "\n\n".join(
        f'<example score="{score}" max_score="{max_score}">\n{answer}\n</example>'
        for score, answer in _normalize_examples(row["examples"])
    )
    user_text = f"""<question id="{row["id"]}" max_score="{max_score}">
{row["question"]}
</question>

<reference_answer>
{row["desired_answer"]}
</reference_answer>

<examples>
{examples_block}
</examples>

<student_answer>
{row["student_answer"]}
</student_answer>

Score the student answer as an integer from 0 to {max_score}. Return JSON only."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

dataset = Dataset.from_list([
    {
        **row,
        "examples": [f"Score {s}/{row['max_score']}: {a}" for s, a in row["examples"]],
        "prompt": make_prompt(row),
        "completion": [{
            "role": "assistant",
            "content": json.dumps({"score": row["student_score"]}),
        }],
    }
    for row in records
])

print(dataset[0].keys())
print(dataset[0]["prompt"][1]["content"])
print(dataset[0]["completion"])

dict_keys(['row_index', 'id', 'question', 'desired_answer', 'student_answer', 'max_score', 'student_score', 'examples', 'prompt', 'completion'])
<question id="1.1" max_score="1">
What is the role of a prototype program in problem solving?
</question>

<reference_answer>
To simulate the behaviour of portions of the desired software product.
</reference_answer>

<examples>
<example score="1" max_score="1">
A prototype is a stripped-down program that mimics how certain parts of the final software will behave, so you can try out ideas early.
</example>

<example score="0" max_score="1">
It is the finished program that you give to the customer.
</example>
</examples>

<student_answer>
High risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.
</student_answer>

Score the student answer as an integer from 0 to 1. Return JSON only.
[{'role': 'assistant', 'conte

## Split by question

- Shuffles unique stripped question strings with `SPLIT_SEED`.
- Allocates unique questions to 80% train, 10% validation, and the remaining test data; row proportions may differ.
- Assigns every row to the split for its question and rejects empty or overlapping splits.

In [8]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))
random.Random(SPLIT_SEED).shuffle(unique_questions)
n_questions = len(unique_questions)
train_end = int(n_questions * 0.80)
validation_end = train_end + int(n_questions * 0.10)
question_splits = {
    "train": set(unique_questions[:train_end]),
    "validation": set(unique_questions[train_end:validation_end]),
    "test": set(unique_questions[validation_end:]),
}
if any(not questions for questions in question_splits.values()):
    raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
assert sum(map(len, question_splits.values())) == len(set.union(*question_splits.values()))
question_to_split = {q: name for name, questions in question_splits.items() for q in questions}
split_indices = {name: [] for name in question_splits}
for index, question in enumerate(question_keys):
    split_indices[question_to_split[question]].append(index)
splits = {name: dataset.select(indices) for name, indices in split_indices.items()}
train_dataset, validation_dataset, test_dataset = (splits[name] for name in ("train", "validation", "test"))
for name, split in splits.items():
    print(f"{name}: {len(split)} rows / {len(question_splits[name])} questions")

train: 3113 rows / 76 questions
validation: 269 rows / 9 questions
test: 396 rows / 10 questions


## Tokenizer

- Loads the Qwen tokenizer and assigns the EOS token as the pad token when needed.
- Uses right padding for both supervised examples and inference.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer:", tokenizer.__class__.__name__)

Tokenizer: Qwen3_5Tokenizer


## Tokenize and verify completion masks

- Applies the same chat template with thinking disabled to the prompt and prompt-plus-completion conversations.
- Verifies that the generation prompt is an exact prefix of the full conversation.
- Masks prompt labels with `-100` and supervises the assistant completion, including its special tokens.
- Measures training/validation lengths before model loading and rejects examples above an explicit `MAX_SEQ_LENGTH`.

In [10]:
def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


formatted_train, formatted_validation = (
    split.map(tokenize_example, remove_columns=split.column_names, desc=f"Tokenize {name}")
    for name, split in (("train", train_dataset), ("validation", validation_dataset))
)
lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
measured_max_length = max(lengths)
sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
if sequence_limit < measured_max_length:
    raise ValueError(
        f"{sum(length > sequence_limit for length in lengths)} examples exceed {sequence_limit} tokens; "
        f"longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
    )
print("Training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
print("Sequence limit:", sequence_limit)
print("Example supervised completion:", tokenizer.decode(
    [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
))

Tokenize train:   0%|          | 0/3113 [00:00<?, ? examples/s]

Tokenize validation:   0%|          | 0/269 [00:00<?, ? examples/s]

Training/validation token lengths: {'count': 3382.0, 'mean': 404.3, 'std': 158.1, 'min': 229.0, '25%': 288.0, '50%': 362.0, '75%': 457.0, 'max': 1056.0}
Sequence limit: 1056
Example supervised completion: {"score": 2}<|im_end|>



## Prediction helper

- Parses a JSON object containing numeric `score` and string `feedback`, enforcing the score range defined by the example's maximum score.
- Rejects non-object output, missing keys, booleans, nonnumeric scores, out-of-range scores, and non-string feedback.
- If surrounding text prevents direct JSON parsing, recovers the last valid JSON object and records that recovery separately.
- Generates greedily with the shared prompt, disabled thinking, and `FEWSHOT_MAX_NEW_TOKENS`.

In [11]:
FEWSHOT_MAX_NEW_TOKENS = 10

def parse_fewshot_prediction(raw_output, max_score):
    recovered = False
    try:
        result = json.loads(raw_output)
    except json.JSONDecodeError:
        # Fall back to the last {...} block in the output (e.g. stray text around the JSON)
        candidates = re.findall(r"\{[^{}]*\}", raw_output, flags=re.DOTALL)
        result = None
        for candidate in reversed(candidates):
            try:
                result = json.loads(candidate)
                break
            except json.JSONDecodeError:
                continue
        recovered = result is not None

    error = None
    if not isinstance(result, dict):
        error = "Output was not a JSON object."
    elif "score" not in result:
        error = "Missing 'score' key."
    elif isinstance(result["score"], bool) or not isinstance(result["score"], (int, float)):
        error = "Score must be a number."
    elif not 0 <= result["score"] <= max_score:
        error = f"Score {result['score']} outside allowed range 0-{max_score}."
    

    return {
        "valid": error is None,
        "raw_output": raw_output,
        "score": result["score"] if error is None else None,
        "error": error,
        "recovered": recovered,
    }


@torch.inference_mode()
def predict_fewshot_score(row):
    inputs = tokenizer.apply_chat_template(
        make_prompt(row),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(model.get_input_embeddings().weight.device)
    generated = model.generate(
        **inputs, do_sample=False,
        max_new_tokens=FEWSHOT_MAX_NEW_TOKENS,  # e.g. 200-300 to leave room for feedback
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_fewshot_prediction(raw_output, max_score=row["max_score"])

## Evaluate the held-out test set

- Generates scalar score-and-feedback predictions for every held-out example using the current model state.
- Counts invalid outputs as incorrect for exact, within-half, and within-one score accuracy.
- Computes MAE, RMSE, normalized MAE, Pearson correlation, and quadratic weighted kappa from valid predictions only.
- Reports valid, invalid, and recovered-output rates together with per-example prediction details.

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    ),
)
model.eval()

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [17]:


for name in ["model", "generated", "inputs"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")

Free: 5.3 GB / 47.7 GB


In [14]:
def evaluate_fewshot_model(dataset, model_name):
    """Generate few-shot predictions and calculate grading metrics for one model state."""
    invalid_count = recovered_count = 0
    exact = within_one = 0
    errors, norm_errors = [], []
    true_scores, pred_scores = [], []
    rows = []
    total = len(dataset)

    for completed, example in enumerate(dataset, start=1):
        true_score = int(example["student_score"])
        max_score = int(example["max_score"])
        prediction = predict_fewshot_score(example)

        is_valid = prediction["valid"]
        recovered_count += int(prediction["recovered"])
        invalid_count += int(not is_valid)

        if is_valid:
            pred_score = int(prediction["score"])
            error = abs(pred_score - true_score)
            errors.append(error)
            norm_errors.append(error / max_score)
            exact += int(error == 0)
            within_one += int(error <= 1)
            true_scores.append(true_score)
            pred_scores.append(pred_score)
        else:
            pred_score = error = None

        rows.append({
            "model": model_name,
            "row_index": example["row_index"],
            "question_id": example["id"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "valid": is_valid,
            "max_score": max_score,
            "true_final_score": true_score,
            "predicted_final_score": pred_score,
            "final_score_absolute_error": error,
            "normalized_absolute_error": error / max_score if is_valid else None,
        })
        if completed % 10 == 0:
            print(f"Completed {completed}/{total} evaluation rows", flush=True)

    n_valid = len(errors)

    if n_valid > 1 and len(set(true_scores)) > 1 and len(set(pred_scores)) > 1:
        pearson = float(np.corrcoef(true_scores, pred_scores)[0, 1])
        qwk = float(cohen_kappa_score(true_scores, pred_scores, weights="quadratic"))
    else:
        pearson = qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": n_valid,
        "recovered_output_rate": recovered_count / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        # Invalid generations count as incorrect for the accuracies.
        "final_score_exact_accuracy": exact / total if total else 0.0,
        "final_score_within_one_accuracy": within_one / total if total else 0.0,
        # The metrics below describe valid predictions only.
        "final_score_mae": sum(errors) / n_valid if n_valid else float("nan"),
        "final_score_rmse": (sum(e ** 2 for e in errors) / n_valid) ** 0.5 if n_valid else float("nan"),
        "normalized_mae": sum(norm_errors) / n_valid if n_valid else float("nan"),
        "final_score_pearson": pearson,
        "final_score_qwk": qwk,
    }
    return metrics, rows


baseline_fewshot_metrics, baseline_fewshot_rows = evaluate_fewshot_model(
    test_dataset, "Baseline Qwen3.5-9B (fewshot)"
)
display(pd.Series(baseline_fewshot_metrics))

NameError: name 'model' is not defined

In [ ]:

results_dir = Path("./evaluation_results")
results_dir.mkdir(parents=True, exist_ok=True)


# Export one row per evaluated example
baseline_rows_df = pd.DataFrame(baseline_fewshot_rows)
baseline_rows_path = results_dir / f"baseline_fewshot_rows.csv"
baseline_rows_df.to_csv(baseline_rows_path, index=False)

# Export aggregate metrics as one CSV row
baseline_metrics_df = pd.DataFrame([baseline_fewshot_metrics])
baseline_metrics_path = results_dir / f"baseline_fewshot_metrics.csv"
baseline_metrics_df.to_csv(baseline_metrics_path, index=False)

print("Saved rows to:", baseline_rows_path)
print("Saved metrics to:", baseline_metrics_path)

In [ ]:
def format_metric(name, value):
    if isinstance(value, (float, np.floating)):
        if "accuracy" in name or "rate" in name:
            return f"{value:.2%}"
        return f"{value:.4f}"
    return str(value)


metrics_table = pd.DataFrame([
    {
        "Metric": name.replace("_", " ").title(),
        "Value": format_metric(name, value),
    }
    for name, value in baseline_fewshot_metrics.items()
])

display(
    metrics_table.style
    .hide(axis="index")
    .set_caption("Baseline Few-Shot Evaluation Metrics")
    .set_properties(**{"text-align": "left"})
)

## Inspect predictions

- Displays the first evaluated rows, including raw output, recovery and validation status, parsing errors, true and predicted final scores, feedback, and score errors.

In [ ]:

evaluation_df = pd.DataFrame(baseline_fewshot_rows)
display(evaluation_df.head(20))